# aw_05_b1b2 — Stages B1/B2: Phase-1 general SFT, two arms (Track B, RQ1+RQ2)

**Protocol**: §5.1 B1/B2. Two arms differing ONLY in response provenance:
- **B1**: curated corpus (`data/p1/p1_general_sft.jsonl`)
- **B2**: rejection-sampled corpus — base-model samples gated by ExactAnswerVerifier
  (`scripts/build_p1_rs_data.py`), same prompt set.

Readouts per arm: (1) P1 general held-out accuracy (`run_p1_eval.py`, retention anchor
is the BASE model's score), (2) **transfer probe** — frozen 200-step PlayWorld SFT
(`probe_playworld_sft.yaml`) evaluated on the frozen suites; probe eval-ID goal-valid
accuracy is the Phase-1 champion primary metric (§6).

**Repos**: B1 → `m97j/aw-runs-b1`, B2 → `m97j/aw-runs-b2`; probe runs get their own
repos `m97j/aw-runs-b1-probe`, `m97j/aw-runs-b2-probe` (fetch_run reads repo-root
artifacts).

**v0.6.2 additions (x09 termination audit, hypothesis T)**:
- `x09a_prereq` — lazy-materialize P1 data (local → HF dataset repo → rebuild guidance).
- `x09b_label_audit` — **GPU-free GATE before training**: rebuilds labels through the
  REAL trainer factory (tiny random model) and checks whether the terminal
  `<|im_end|>`/EOS label is masked (-100) under `pad_token == eos_token`. If
  CONFIRMED, do NOT train — fix `models/builder.py` padding policy first.
- `c_b12_p1_eval` now passes `--dump-predictions` (per-row jsonl, synced to HF via
  the same `p1_eval/` upload) feeding `x09d_drift_audit`.
- `x09c_run_audit` — audits probe-eval run artifacts (`truncated_outputs`, verdict
  reason codes, runaway-tail heuristic).
- `x09d_drift_audit` — GSM8K answer-extraction drift using the verifier's own
  `extract_final_answer` (marker-anchored vs naive last-number vs gold).
- `x10_stop_logit_probe` — after x09b rejected hypothesis T, discriminates
  U1 (stop-logit under-training) vs U2 (train/eval rendering mismatch) via a
  teacher-forcing probe of P(<|im_end|>) at the gold stop position, base vs adapter,
  train-rendering vs eval-rendering.

Session-resume policy everywhere: local reuse → HF fetch → loud failure with the
exact prerequisite stage to run. PlayWorld data/suites are cheap → always rebuilt.

In [ ]:
# @title common header
import os

from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get('HF_TOKEN')
os.environ["WANDB_API_KEY"] = userdata.get('WANDB_API_KEY')
os.environ["GITHUB_TOKEN"] = userdata.get('GITHUB_TOKEN')

!git clone https://{os.environ["GITHUB_TOKEN"]}@github.com/m97j/axiom-world.git
%cd axiom-world
!pip install -e . -r requirements/colab-g4.lock.txt

In [ ]:
# @title x09a_prereq — materialize x09/eval inputs (local -> HF -> rebuild/fail)
from pathlib import Path

HF_DATASET_REPO = "m97j/axiom-general-posttrain"  # dataset repo used by a_b12_data syncs
P1_PATH_IN_REPO = "p1/v1"  # subdir inside the dataset repo holding p1 files ("" = repo root)

P1_TRAIN   = Path("data/p1/p1_general_sft.jsonl")      # x09b (check C) input
P1_HOLDOUT = Path("data/p1/p1_general_holdout.jsonl")  # run_p1_eval input
P1_FILES = ("p1_general_sft.jsonl", "p1_general_holdout.jsonl", "p1_manifest.json")


def materialize_p1() -> None:
    if P1_TRAIN.exists() and P1_HOLDOUT.exists():
        print("[x09a] p1 data: LOCAL reuse")
        return
    try:
        from huggingface_hub import hf_hub_download
        P1_TRAIN.parent.mkdir(parents=True, exist_ok=True)
        for name in P1_FILES:
            rel = f"{P1_PATH_IN_REPO}/{name}" if P1_PATH_IN_REPO else name
            got = hf_hub_download(HF_DATASET_REPO, rel, repo_type="dataset")
            (P1_TRAIN.parent / name).write_bytes(Path(got).read_bytes())
        assert P1_TRAIN.exists() and P1_HOLDOUT.exists()
        print("[x09a] p1 data: fetched from HF dataset repo")
    except Exception as exc:  # noqa: BLE001 — surface guidance, then fail loudly
        raise SystemExit(
            "[x09a] p1 data missing locally AND on HF dataset repo "
            f"({HF_DATASET_REPO}).\n"
            "Run stage a_b12_data first (python scripts/build_p1_data.py ...).\n"
            "WARNING: rebuilding may change the frozen holdout fingerprint — "
            "verify p1_manifest.json holdout_fingerprint against the protocol."
        ) from exc


materialize_p1()

# PlayWorld artifacts are cheap to regenerate -> never persisted, always rebuildable
if not Path("data/eval_suites").exists():
    !python scripts/build_eval_suites.py --episodes-per-suite 300
if not Path("data/training").exists():
    !python scripts/build_training_data.py

In [ ]:
# @title x09b_label_audit — hypothesis-T GATE via the REAL trainer factory (CPU, no GPU needed)
# Runs BEFORE any (re)training: if the terminal stop-token label is masked in the
# collated batch under pad_id == eos_id, SFT arms cannot learn to terminate and
# further training spend is wasted until models/builder.py padding policy is fixed.
!python scripts/x09_termination_audit.py \
  --sft-config configs/experiments/b1_general_sft.yaml \
  --sft-jsonl data/p1/p1_general_sft.jsonl \
  --n-label-samples 4 \
  --out runs/x09_label_audit.json

import json as _json

_verdict = _json.load(open("runs/x09_label_audit.json"))["label_audit"]["verdict"] # noqa: SIM115
print("\n[x09b VERDICT]", _verdict)
HYPOTHESIS_T_CONFIRMED = _verdict.startswith("HYPOTHESIS-T CONFIRMED")
if HYPOTHESIS_T_CONFIRMED:
    print(
        "[x09b] GATE: SKIP b_b1_train / b_b2_train — fix models/builder.py padding "
        "policy (distinct pad token) + protocol v1.2 amendment, then retrain."
    )

In [ ]:
# @title a_b12_data — P1 mixture + held-out + RS corpus (skip pieces already materialized)
!python scripts/build_p1_data.py
# base-model reference on the held-out (retention anchor, run once; left-padding fixed)
!python scripts/run_p1_eval.py \
  --config configs/experiments/b1_general_sft.yaml \
  --holdout data/p1/p1_general_holdout.jsonl \
  --label qwen3-8b-base --output runs/p1_eval_base.json \
  --dump-predictions runs/p1_eval_base_predictions.jsonl \
  --hf-sync-repo m97j/aw-runs-b1
# B2 rejection-sampled corpus (generation-heavy; skip if already on the dataset repo)
!python scripts/build_p1_rs_data.py \
  --config configs/experiments/b1_general_sft.yaml \
  --input data/p1/p1_general_sft.jsonl \
  --output data/p1/p1_general_sft_rs.jsonl \
  --num-candidates 4 --batch-size 64 \
  --hf-sync-repo m97j/axiom-general-posttrain

In [ ]:
# @title b_b1_train — P1 general SFT (curated arm)
assert not HYPOTHESIS_T_CONFIRMED, "x09b gate: fix padding policy before training"
!python scripts/run_experiment.py \
  --config configs/experiments/b1_general_sft.yaml \
  --hf-sync-repo m97j/aw-runs-b1

In [ ]:
# @title b_b2_train — P1 general SFT (rejection-sampled arm)
assert not HYPOTHESIS_T_CONFIRMED, "x09b gate: fix padding policy before training"
!python scripts/run_experiment.py \
  --config configs/experiments/b2_general_sft_rs.yaml \
  --hf-sync-repo m97j/aw-runs-b2

In [ ]:
# @title c_b12_p1_eval — held-out accuracy for both arms (retention readout, + per-row dumps)
B1_RUN_ID = "20260803-162417--b1-general-sft--s42--b08a2c"  # <- from b_b1_train "run_id: ..."
B2_RUN_ID = "20260803-165015--b2-general-sft-rs--s42--10ecce"  # <- from b_b2_train "run_id: ..."

out = !python scripts/fetch_run.py --repo m97j/aw-runs-b1 --run-id {B1_RUN_ID}
b1_dir = [line for line in out if line.startswith("ADAPTER_DIR=")][0].split("=", 1)[1]
out = !python scripts/fetch_run.py --repo m97j/aw-runs-b2 --run-id {B2_RUN_ID}
b2_dir = [line for line in out if line.startswith("ADAPTER_DIR=")][0].split("=", 1)[1]

!python scripts/run_p1_eval.py \
  --config configs/experiments/b1_general_sft.yaml \
  --adapter-dir {b1_dir} --label b1-sft \
  --output runs/p1_eval_b1.json \
  --dump-predictions runs/p1_eval_b1_predictions.jsonl \
  --hf-sync-repo m97j/aw-runs-b1
!python scripts/run_p1_eval.py \
  --config configs/experiments/b2_general_sft_rs.yaml \
  --adapter-dir {b2_dir} --label b2-sft-rs \
  --output runs/p1_eval_b2.json \
  --dump-predictions runs/p1_eval_b2_predictions.jsonl \
  --hf-sync-repo m97j/aw-runs-b2

In [ ]:
# @title x09d_drift_audit — GSM8K answer-extraction drift per arm (CPU)
# Consumes the per-row dumps from c_b12_p1_eval. New session: dumps live under
# p1_eval/ in each run repo (same upload_directory sync as the summary json).
from pathlib import Path

for label, repo in (("b1", "m97j/aw-runs-b1"), ("b2", "m97j/aw-runs-b2")):
    pred = Path(f"runs/p1_eval_{label}_predictions.jsonl")
    if not pred.exists():
        try:
            from huggingface_hub import hf_hub_download
            got = hf_hub_download(repo, f"p1_eval/{pred.name}")
            pred.parent.mkdir(parents=True, exist_ok=True)
            pred.write_bytes(Path(got).read_bytes())
            print(f"[x09d] {pred.name}: fetched from HF ({repo})")
        except Exception as exc:  # noqa: BLE001
            raise SystemExit(
                f"[x09d] {pred} not found locally or on {repo}. "
                "Re-run c_b12_p1_eval with --dump-predictions first (GPU stage)."
            ) from exc
    else:
        print(f"[x09d] {pred.name}: LOCAL reuse")
    !python scripts/x09_termination_audit.py \
      --p1-predictions {pred} --out runs/x09_drift_audit_{label}.json

In [ ]:
# @title d_b12_probe — frozen transfer probe (200 steps) per arm
import json
from pathlib import Path

assert not HYPOTHESIS_T_CONFIRMED, "x09b gate: fix padding policy before training"
!python scripts/build_training_data.py
!python scripts/build_eval_suites.py --episodes-per-suite 300


def get_adapter_sha(run_id: str) -> str:
    """Safely extracts the SHA from the run lineage artifact."""
    path = Path("runs") / run_id / "artifacts" / "lineage.json"
    data = json.loads(path.read_text())
    return data["output_adapter_sha256"]

b1_sha = get_adapter_sha(B1_RUN_ID)
b2_sha = get_adapter_sha(B2_RUN_ID)

!python scripts/run_experiment.py \
  --config configs/experiments/probe_playworld_sft.yaml \
  --parent-adapter-dir {b1_dir} \
  --override lineage.parent_adapter.repo_id=m97j/aw-runs-b1 \
  --override lineage.parent_adapter.sha256={b1_sha} \
  --override experiment_name=probe-playworld-sft-b1 \
  --hf-sync-repo m97j/aw-runs-b1-probe

!python scripts/run_experiment.py \
  --config configs/experiments/probe_playworld_sft.yaml \
  --parent-adapter-dir {b2_dir} \
  --override lineage.parent_adapter.repo_id=m97j/aw-runs-b2 \
  --override lineage.parent_adapter.sha256={b2_sha} \
  --override experiment_name=probe-playworld-sft-b2 \
  --hf-sync-repo m97j/aw-runs-b2-probe


In [ ]:
# @title e_b12_probe_eval — probe adapters on the frozen suites
B1_PROBE_RUN = "20260803-173412--probe-playworld-sft-b1--s42--4c6724"  # <- from d_b12_probe outputs
B2_PROBE_RUN = "20260803-174349--probe-playworld-sft-b2--s42--73e35f"

out = !python scripts/fetch_run.py --repo m97j/aw-runs-b1-probe --run-id {B1_PROBE_RUN}
b1p_dir = [line for line in out if line.startswith("ADAPTER_DIR=")][0].split("=", 1)[1]
out = !python scripts/fetch_run.py --repo m97j/aw-runs-b2-probe --run-id {B2_PROBE_RUN}
b2p_dir = [line for line in out if line.startswith("ADAPTER_DIR=")][0].split("=", 1)[1]

!python scripts/run_evaluation.py \
  --config configs/experiments/eval_playworld.yaml \
  --adapter-dir {b1p_dir} \
  --max-new-tokens 1024 --batch-size 100 --hf-sync-repo m97j/aw-runs-b1-probe
!python scripts/run_evaluation.py \
  --config configs/experiments/eval_playworld.yaml \
  --adapter-dir {b2p_dir} \
  --max-new-tokens 1024 --batch-size 100 --hf-sync-repo m97j/aw-runs-b2-probe

In [ ]:
# @title x09c_run_audit — termination audit of probe-eval run artifacts (CPU)
B1_PROBE_EVAL = "20260803-175421--eval-playworld--s42--fd3d91"  # <- eval run ids from e_b12_probe_eval
B2_PROBE_EVAL = "20260803-182912--eval-playworld--s42--0276f7"

# local -> HF (fetch_run --kind eval verifies against the persisted run)
!python scripts/fetch_run.py --repo m97j/aw-runs-b1-probe --run-id {B1_PROBE_EVAL} --kind eval
!python scripts/fetch_run.py --repo m97j/aw-runs-b2-probe --run-id {B2_PROBE_EVAL} --kind eval

!python scripts/x09_termination_audit.py \
  --run-dirs runs/{B1_PROBE_EVAL} runs/{B2_PROBE_EVAL} \
  --out runs/x09_run_audit.json

In [ ]:
# @title x10_stop_logit_probe — U1 (under-training) vs U2 (rendering mismatch) discrimination
# Context: x09b REJECTED hypothesis T (terminal <|im_end|> labels LIVE); x09c shows
# runaway_rate ~0.96-0.98 on probe evals. Teacher-forcing probe of P(<|im_end|>) at
# the gold stop position under TRAIN vs EVAL rendering. Base run = reference delta.
B1_RUN_ID_X10 = "20260803-162417--b1-general-sft--s42--b08a2c"  # <- b_b1_train run id (reuse B1_RUN_ID if set)

out = !python scripts/fetch_run.py --repo m97j/aw-runs-b1 --run-id {B1_RUN_ID_X10}
b1_dir_x10 = [line for line in out if line.startswith("ADAPTER_DIR=")][0].split("=", 1)[1]

# base-model reference (no adapter)
!python scripts/x10_stop_logit_probe.py \
  --config configs/experiments/b1_general_sft.yaml \
  --sft-jsonl data/p1/p1_general_sft.jsonl \
  --num-samples 8 --eval-rendering \
  --out runs/x10_stop_logit_base.json

# B1 adapter
!python scripts/x10_stop_logit_probe.py \
  --config configs/experiments/b1_general_sft.yaml \
  --adapter-dir {b1_dir_x10} \
  --sft-jsonl data/p1/p1_general_sft.jsonl \
  --num-samples 8 --eval-rendering \
  --out runs/x10_stop_logit_b1.json

import json as _json  # noqa: E402

for tag in ("base", "b1"):
    rep = _json.load(open(f"runs/x10_stop_logit_{tag}.json"))  # noqa: SIM115
    print(f"[x10:{tag}] mean_p_im_end={rep['mean_p_im_end']}")
    print(f"[x10:{tag}] VERDICT: {rep['verdict']}\n")
    

In [ ]:
# @title x11_adapter_integrity — explain the bit-identical B1==B2 x10 anomaly (CPU; --forward-diff on GPU)
# x10 returned IDENTICAL logit reports for the B1 and B2 adapters (impossible for
# independently trained LoRAs) and near-uniform garbage distributions. x11 hashes
# both adapter dirs, diffs tensors elementwise, checks all-zero lora_B (no-op
# adapter), and (GPU) measures forward logit distance base vs each adapter.
B1_RUN_ID_X11 = "20260803-162417--b1-general-sft--s42--b08a2c"  # <- b1-general-sft run id
B2_RUN_ID_X11 = "20260803-165015--b2-general-sft-rs--s42--10ecce"  # <- b2-general-sft-rs run id

out = !python scripts/fetch_run.py --repo m97j/aw-runs-b1 --run-id {B1_RUN_ID_X11}
b1_dir_x11 = [line for line in out if line.startswith("ADAPTER_DIR=")][0].split("=", 1)[1]
out = !python scripts/fetch_run.py --repo m97j/aw-runs-b2 --run-id {B2_RUN_ID_X11}
b2_dir_x11 = [line for line in out if line.startswith("ADAPTER_DIR=")][0].split("=", 1)[1]
print("b1:", b1_dir_x11)
print("b2:", b2_dir_x11)
assert b1_dir_x11 != b2_dir_x11, "fetch_run returned the SAME dir for both runs (wiring bug found)"

!python scripts/x11_adapter_integrity.py \
  --adapter-dirs {b1_dir_x11} {b2_dir_x11} \
  --config configs/experiments/b1_general_sft.yaml \
  --forward-diff \
  --out runs/x11_adapter_integrity.json

import json as _json  # noqa: E402

rep = _json.load(open("runs/x11_adapter_integrity.json"))   # noqa: SIM115
print("\n[x11 pairwise]", [p.get("verdict") for p in rep["pairwise"]])
print("[x11 VERDICT]", rep["verdict"])

In [ ]:
# @title f_b12_analysis — probe-vs-probe and probe-vs-A1
A1_EVAL = "20260801-063425--eval-playworld--s42--3bf440"

!python scripts/fetch_run.py --repo m97j/aw-runs-a1 --run-id {A1_EVAL} --kind eval

!python scripts/run_analysis.py \
  --run-a runs/{B1_PROBE_EVAL} --label-a b1-probe \
  --run-b runs/{B2_PROBE_EVAL} --label-b b2-probe \
  --output runs/{B1_PROBE_EVAL}/analysis_b1_vs_b2_probe.json --hf-sync-repo m97j/aw-runs-b1-probe

!python scripts/run_analysis.py \
  --run-a runs/{B1_PROBE_EVAL} --label-a b1-probe \
  --run-b runs/{A1_EVAL} --label-b a1-sft \
  --output runs/{B1_PROBE_EVAL}/analysis_b1probe_vs_a1.json --hf-sync-repo m97j/aw-runs-b1-probe

## Stage checklist (feeds §6 Phase-1 champion selection)
- [ ] **x09b label-audit verdict recorded** (hypothesis T confirmed/rejected — gate honored)
- [ ] base / B1 / B2 held-out accuracies recorded (retention: drop ≤ 3 pts vs base)
- [ ] per-row prediction dumps synced (`p1_eval/p1_eval_{base,b1,b2}_predictions.jsonl`)
- [ ] RS manifest: acceptance_rate + coverage reported
- [ ] probe eval-ID goal-valid accuracy per arm = Phase-1 primary metric input
- [ ] **x09c run-audit**: truncation_rate / runaway_rate per probe eval run recorded
- [ ] **x09d drift-audit**: drift_caused_failures ≈ 0 confirms collapse is not extraction
- [ ] Winner(s) proceed to B3 (P1 DPO, aw_06)